# SmartNest Security Assessment Plan

This notebook records the assessment plan for the SmartNest coursework project.

Project focus:
- containerised smart home simulation
- threat modelling and attack demonstration
- defensive redesign and validation

Environment assumption:
All components run in Docker containers on a local development machine rather than on Raspberry Pi or ESP32 hardware.


---

## 整体场景设定

模拟一个 **"SmartNest" 智能家居平台**，包含以下组件：

**中心 Hub**：Docker 容器中的 MQTT Broker + Web 管理面板
**设备节点**：Python 模拟设备容器，分别模拟智能门锁、智能灯和智能闹钟
**通信协议**：MQTT（设备↔Broker）、HTTP/HTTPS（用户↔Web 面板）
**部署方式**：通过 Docker Compose 在本地电脑上完成一键编排和复现实验环境

---

## Coursework 1：威胁建模 & 攻击模拟

### 威胁 1：MQTT 中间人攻击（MitM on MQTT）

**原理**：很多智能家居系统仍使用未加密的 MQTT（端口 1883），攻击者一旦能够访问相同网络或宿主机开放端口，就可以嗅探和伪造消息。

**具体实现步骤**：
1. 在 Docker 中部署 Mosquitto Broker，**故意不开 TLS**，并开放 1883 端口
2. 设备模拟器容器持续发布状态到 `home/lock/status`、`home/light/status`、`home/alarm/status`，并订阅对应 command topic
3. 攻击者在宿主机或另一容器中使用 `mosquitto_sub` 和 `mosquitto_pub` 监听与伪造 MQTT 消息
4. 用 Wireshark 或 `tcpdump` 抓包，展示明文 MQTT payload（例如开锁指令、设备状态、用户名字段）
5. 用 `mosquitto_pub` 伪造一条开锁指令：`mosquitto_pub -h localhost -t home/lock/command -m '{"action":"UNLOCK"}'`，观察设备状态变化

**交付物**：Wireshark 截图、攻击脚本、攻击链流程图

### 威胁 2：Web 管理面板漏洞（认证绕过 + 注入）

**原理**：智能家居 Hub 通常有一个 Web 管理界面，如果开发不当，容易存在弱认证、SQL 注入、XSS 等问题。

**具体实现步骤**：
1. 在 Docker 容器中运行 Flask 管理面板（设备列表、用户登录、设备控制页面）
2. **故意留漏洞**：默认密码 admin/admin、SQL 注入点（登录表单）、未做 CSRF 保护的设备控制接口
3. 演示攻击：用 `sqlmap` 跑注入拿到用户表、用默认密码登录、通过 CSRF 构造恶意页面远程控制设备
4. 可选加分项：演示 XSS 存储型攻击——在设备名称字段注入脚本，管理员查看时触发

**交付物**：漏洞利用脚本、sqlmap 输出截图、CSRF PoC HTML 页面

### 威胁 3（可选）：容器逃逸 / 镜像配置暴露

可以进一步分析 Docker 镜像、Compose 配置和容器间网络暴露面，例如弱默认配置、明文环境变量、过宽的端口暴露或缺少认证的内部 API。

### 风险矩阵

用一个 5×5 的风险矩阵（Likelihood × Impact）对威胁排优先级，MQTT MitM 是"高可能 × 高影响"，Web 漏洞是"中可能 × 高影响"。

---

## Coursework 2：防御方案

### 防御 1：MQTT over TLS + 设备证书认证

1. 给 Mosquitto 配置 TLS（生成 CA 证书、服务端证书、客户端证书）
2. 为每个设备模拟客户端分配唯一客户端证书，Broker 开启 `require_certificate true`
3. 演示：攻击者再次尝试嗅探 → Wireshark 只看到加密流量；伪造消息 → 被 Broker 拒绝连接
4. 代码展示证书生成脚本和 Python MQTT 客户端的 TLS 连接配置

### 防御 2：AI 驱动的入侵检测系统（IDS）

1. 在 Hub 上部署一个轻量 Python IDS 脚本
2. 用正常运行时的 MQTT 流量训练一个基线模型（可以用 Isolation Forest 或简单的统计阈值）
3. 检测异常：突然大量 publish 请求、非正常时间段的开锁指令、陌生 Client ID 连接
4. 触发告警（邮件/Telegram 通知 + Web 面板红色警告）
5. 展示对比：无 IDS 时攻击成功 vs 有 IDS 时 30 秒内检测到并阻断

### 防御 3：Web 面板加固

1. 用参数化查询消除 SQL 注入
2. 实现 CSRF Token
3. 加入 MFA（TOTP，用 `pyotp` 库实现）
4. 演示：加固后再跑 sqlmap → 无果；没有 TOTP 码无法登录

### 防御 4：安全 OTA 固件更新

1. 将其调整为**安全镜像/配置更新机制**：对容器镜像或更新包进行签名校验（用 RSA/Ed25519）
2. 演示：篡改过的更新包或配置文件被系统拒绝加载

---

## 4 人分工

| 成员 | 负责模块 | 具体任务 |
|------|---------|---------|
| A | MQTT 攻防 | 搭建 MQTT 环境、MitM 攻击演示、TLS 防御实现 |
| B | Web 安全 | 搭建 Flask 面板、留漏洞+攻击演示、加固+MFA |
| C | IDS 系统 | 流量采集、异常检测模型训练、告警系统 |
| D | 架构 + 集成 | Docker 架构设计、Compose 编排、安全更新机制、报告整合 |

每个人的工作互相依赖（比如 C 的 IDS 要接 A 的 MQTT 流量），但又可以独立开发测试，最后集成。

---

## 实验环境与资源需求

本项目不依赖树莓派、ESP32 或其他实体硬件。实验环境基于本地电脑上的 Docker Desktop / Docker Engine 搭建，包含 Mosquitto 容器、Web 面板容器和 Python 设备模拟器容器。攻击端可直接使用宿主机工具，或者额外启一个攻击容器来复现监听、伪造消息和漏洞利用流程。

---



为了让项目更容易复现、演示和提交，我们统一采用 **Docker 容器模拟环境**，不再依赖树莓派或 ESP32 实体硬件。下面是一套更适合当前 repo 的容器化实施方案。

---

## 需要准备什么


---

## 容器架构

整体结构是这样的：Docker Compose 创建一个内部网络，其中 Mosquitto 作为 Broker，Flask 作为 Web 面板，Python 脚本作为设备模拟器。攻击者既可以从宿主机通过映射端口接入，也可以加入同一个 Docker 网络。这样能较稳定地重现“弱认证 Web 面板 + 明文 MQTT + 可伪造设备指令”的攻击链。

---

## 具体搭建步骤

### 第一步：配置容器化 Hub

在本地电脑上准备 `docker-compose.yml`，定义 Mosquitto Broker、Flask Web 面板和设备模拟器三个服务。Broker 初始配置保持不安全：不开 TLS、不设 MQTT 用户认证、开放 1883 端口，方便 Coursework 1 做监听和伪造攻击。

记录对外暴露的端口，例如 Web 面板 `localhost:5001`、MQTT Broker `localhost:1883`，后续攻击和演示都围绕这些接口展开。

### 第二步：实现设备模拟器

用 Python 编写一个设备模拟器进程或容器，模拟多个智能设备的状态机和命令处理逻辑。

**门锁模拟器：** 订阅 MQTT topic `home/lock/command`，收到 `"UNLOCK"` 就把状态切换为 `UNLOCKED`，并发布到 `home/lock/status`。

**其他设备模拟器：** 智能灯和智能闹钟通过相同方式运行，周期性上报状态，并根据控制指令更新内部状态。

核心逻辑就是 MQTT 连接 → 订阅 command topic → 循环发布 status / event topic。这样既保留 IoT 行为模型，也更容易调试和复现实验。

### 第三步：搭建 Web 管理面板

在单独的 Flask 容器中实现一个简单的管理界面。首页显示所有设备的实时状态，有按钮可以发送控制指令，并提供登录页面。这个面板就是 Coursework 1 里被攻击的目标之一，也是 Coursework 2 里被加固的对象。

### 第四步：联调验收

启动全部容器后，打开三个终端窗口分别观察：一个看 Mosquitto 日志，一个用 `mosquitto_sub -t 'home/#'` 订阅所有 MQTT 消息，一个打开浏览器访问 Flask 面板。在面板上点“开锁”后，验证设备模拟器状态是否变化，并确认新的状态消息已经被 Web 面板接收。这条链路打通后，环境就搭好了。

---

## 为什么选择 Docker

对 coursework 来说，Docker 最大的优势是**可复现**、**易分享** 和 **易重置**。所有成员都可以在自己的电脑上拉起同样的环境，不会被硬件借用、网络配置或烧录问题卡住。

另外，Docker 也更适合做攻击与防御前后的对比：可以快速替换 broker 配置、切换 insecure / hardened 版本、重放攻击脚本，并保留日志和抓包证据，便于写报告和录视频。

---


下面是一个整体的协议架构图，然后列出每个设备的详细 JSON 规范。每个设备遵循统一的三类 topic：`status`（设备上报状态）、`command`（面板下发指令）、`event`（设备上报事件告警）。下面是完整 JSON 规范。

---

## 公共字段（所有消息都带）

```json
{
  "device_id": "lock_01",
  "device_type": "lock",
  "timestamp": 1712200000,
  "version": "1.0"
}
```

`device_type` 固定三个值：`"lock"` / `"light"` / `"alarm"`。`version` 是协议版本号，方便以后扩展。

---

## 1. 智能门锁 (Smart Lock)

**`home/lock/status`** — 门锁每 10 秒上报一次

```json
{
  "device_id": "lock_01",
  "device_type": "lock",
  "timestamp": 1712200000,
  "version": "1.0",
  "state": "LOCKED",
  "battery": 85,
  "method": "auto",
  "last_user": "admin"
}
```

`state` 取值：`"LOCKED"` / `"UNLOCKED"`
`method` 记录最近一次操作方式：`"manual"` / `"remote"` / `"auto"` / `"pin"`
`last_user` 记录最近操作人

**`home/lock/command`** — Web 面板下发

```json
{
  "device_id": "lock_01",
  "device_type": "lock",
  "timestamp": 1712200020,
  "version": "1.0",
  "action": "UNLOCK",
  "issued_by": "admin",
  "pin": "1234"
}
```

`action` 取值：`"LOCK"` / `"UNLOCK"` / `"SET_PIN"`
`pin` 仅在 `SET_PIN` 时必填，其他时候可选（用于验证身份）

**`home/lock/event`** — 异常事件触发时发送

```json
{
  "device_id": "lock_01",
  "device_type": "lock",
  "timestamp": 1712200030,
  "version": "1.0",
  "event": "TAMPER_DETECTED",
  "severity": "high",
  "detail": "Vibration sensor triggered"
}
```

`event` 取值：`"TAMPER_DETECTED"` / `"LOW_BATTERY"` / `"FORCED_ENTRY"` / `"PIN_FAILED"`
`severity`：`"low"` / `"medium"` / `"high"` / `"critical"`

---

## 2. 智能灯 (Smart Light)

**`home/light/status`** — 每 10 秒上报

```json
{
  "device_id": "light_01",
  "device_type": "light",
  "timestamp": 1712200000,
  "version": "1.0",
  "power": "ON",
  "brightness": 80,
  "color_temp": 4000,
  "rgb": [255, 180, 100],
  "mode": "manual"
}
```

`power`：`"ON"` / `"OFF"`
`brightness`：0-100 整数
`color_temp`：色温（开尔文），2700 暖光 ~ 6500 冷光
`rgb`：RGB 值，和 `color_temp` 二选一使用
`mode`：`"manual"` / `"schedule"` / `"auto"`（自动亮度）

**`home/light/command`** — 控制指令

```json
{
  "device_id": "light_01",
  "device_type": "light",
  "timestamp": 1712200020,
  "version": "1.0",
  "action": "SET_BRIGHTNESS",
  "issued_by": "admin",
  "params": {
    "brightness": 50
  }
}
```

`action` 取值：`"ON"` / `"OFF"` / `"SET_BRIGHTNESS"` / `"SET_COLOR"` / `"SET_MODE"`
`params` 根据 action 不同而变化：
- `SET_BRIGHTNESS` → `{"brightness": 50}`
- `SET_COLOR` → `{"rgb": [255, 0, 0]}` 或 `{"color_temp": 3000}`
- `SET_MODE` → `{"mode": "schedule", "schedule": {"on": "18:00", "off": "23:00"}}`

**`home/light/event`** — 异常事件

```json
{
  "device_id": "light_01",
  "device_type": "light",
  "timestamp": 1712200030,
  "version": "1.0",
  "event": "OVERHEATING",
  "severity": "high",
  "detail": "Temperature reached 85C, auto shutdown"
}
```

`event` 取值：`"OVERHEATING"` / `"BULB_FAILURE"` / `"POWER_SURGE"`

---

## 3. 智能闹钟 (Smart Alarm Clock)

**`home/alarm/status`** — 每 10 秒上报

```json
{
  "device_id": "alarm_01",
  "device_type": "alarm",
  "timestamp": 1712200000,
  "version": "1.0",
  "alarm_time": "07:30",
  "enabled": true,
  "ringing": false,
  "volume": 70,
  "repeat": ["MON", "TUE", "WED", "THU", "FRI"],
  "snooze_count": 0
}
```

`alarm_time`：24 小时制字符串
`repeat`：重复日期数组，空数组 `[]` 表示仅一次
`snooze_count`：当前已贪睡的次数

**`home/alarm/command`** — 控制指令

```json
{
  "device_id": "alarm_01",
  "device_type": "alarm",
  "timestamp": 1712200020,
  "version": "1.0",
  "action": "SET_ALARM",
  "issued_by": "admin",
  "params": {
    "alarm_time": "06:45",
    "repeat": ["MON", "WED", "FRI"],
    "volume": 80
  }
}
```

`action` 取值：`"SET_ALARM"` / `"ENABLE"` / `"DISABLE"` / `"DISMISS"` / `"SNOOZE"`
- `DISMISS` 和 `SNOOZE` 不需要 `params`
- `SNOOZE` 默认延迟 5 分钟（设备端硬编码）

**`home/alarm/event`** — 事件通知

```json
{
  "device_id": "alarm_01",
  "device_type": "alarm",
  "timestamp": 1712200030,
  "version": "1.0",
  "event": "ALARM_TRIGGERED",
  "severity": "low",
  "detail": "Alarm ringing at 07:30"
}
```

`event` 取值：`"ALARM_TRIGGERED"` / `"ALARM_MISSED"` / `"MAX_SNOOZE_REACHED"`

---

## 为什么这样设计

统一的三 topic 模式（status / command / event）让每个设备的行为可预测，IDS 做异常检测时规则统一。`params` 字段把不同指令的参数嵌套在一起，避免顶层字段爆炸。`severity` 字段让告警系统可以按严重程度分级处理。

攻击演示的时候，这套 JSON 格式在 Wireshark 里抓包特别直观——攻击者一眼就能看到 `"action": "UNLOCK"` 和 `"issued_by": "admin"`，展示明文传输的危险性。


现在环境跑通了，CW1 的核心是**威胁建模 + 攻击演示**，按报告模板（Introduction、Threat Model、Impact & Prioritization、Data Sources & Attack Setup）来推进。下面是接下来一周的具体路线。

---


报告里 60% 的分都看攻击有多扎实，所以先把演示做出来，之后所有内容都围绕它写。

按 Sec_Plan，CW1 要演示**两个威胁**。建议两个组员并行：

### 攻击 A：MQTT 嗅探与指令伪造

1. **被动嗅探截图** — 开 Wireshark 选 `lo0` 接口，过滤 `mqtt`，截一张能看到明文 JSON payload 的图（最好包含 `"action":"UNLOCK"` 那一包）
2. **指令伪造录屏** — 分屏录制：左边浏览器 Dashboard 显示 `LOCKED`，右边终端跑 `mosquitto_pub` 发 UNLOCK 指令，浏览器立刻变 `UNLOCKED` 而且 `Last user` 变成 `attacker`
3. **攻击脚本** — 整理成 `attacks/mqtt_attack.sh`，带注释，方便老师复现

### 攻击 B：Web 漏洞利用

1. **SQL 注入登录绕过** — 在登录页用户名输入 `admin' --`，密码随便填，直接登录成功。截图保存
2. **sqlmap 拖库** — 跑 `sqlmap -u "http://localhost:5001/login" --data="username=a&password=b" --dump`，截一张拿到 users 表的图
3. **CSRF PoC** — 写一个 `attacks/csrf.html`，里面是个隐藏表单自动 POST 到 `/command` 发 UNLOCK，演示用户访问恶意页面就能开锁
4. **未认证 API** — 直接 curl `http://localhost:5001/api/devices` 拿到所有设备数据，截图

---

## 第 2 步：写威胁模型

用 **STRIDE** 框架对每个威胁分类（Spoofing / Tampering / Repudiation / Information Disclosure / DoS / Elevation of Privilege）。每个威胁写：

- 攻击面（哪个组件、哪个端口、哪个 endpoint）
- 攻击者画像（外部、内网、内鬼）
- 前置条件
- 攻击步骤（对应你们演示的命令）
- 影响

最后画一张 **5×5 风险矩阵**（Likelihood × Impact），把两个威胁标上去，给出优先级排序。

---

## 第 3 步：架构图 + 资产清单

画一张**系统架构图**（可以用 draw.io 或 Excalidraw），标出三个容器、MQTT/HTTP 数据流、对外暴露的端口（1883、5001）。然后加一个**关键资产表**：

| 资产 | 类别 | 敏感度 | 位置 |
|------|------|--------|------|
| 用户凭据 | 数据 | 高 | SQLite users 表 |
| 门锁状态/指令 | 数据 | 高 | MQTT topic |
| 用户隐私（作息） | 数据 | 中 | alarm_time 字段 |
| Web 管理面板 | 应用 | 高 | Flask 服务 |
| MQTT Broker | 基础设施 | 高 | Mosquitto |

---

## 第 4 步：整合到报告

报告 5 页非常紧，按这个比例分配：

- **Introduction & objectives** — 半页（场景介绍 + 目标）
- **Threat model** — 1.5 页（架构图 + STRIDE 表 + 威胁详细描述）
- **Impact & prioritization** — 1 页（风险矩阵 + 排序理由）
- **Data sources & attack setup** — 1.5 页（环境搭建 + 两个攻击的步骤和截图）
- **小结** — 半页（引出 CW2 防御方案）

图要多，文字要精炼。每个攻击至少配一张截图。

---




## 核心思路：防御 vs 威胁对应表

先把 CW2 的 4 个防御和 CW1 的 5 个威胁对齐，让报告叙事紧凑：

| 防御 | 解决的威胁 | 负责人 | 工作量 |
|------|-----------|--------|--------|
| D1 — MQTT TLS + 客户端证书 | T1 + T2 | A | 中（证书生成有坑） |
| D2 — AI 驱动 IDS | T2 残余风险 + T4 + flood | C | 大（要训练模型） |
| D3 — Web 加固（参数化查询 + CSRF + MFA） | T3 + T4 + T5 | B | 中 |
| D4 — 安全镜像/配置签名 | 补充威胁（供应链） | D | 小（演示为主） |

**这样 5 个威胁全部被覆盖**，而且 D2 的 IDS 正好是"纵深防御"思想的体现——即使 D1/D3 被绕过，IDS 仍能报警。

---


**D1 — MQTT TLS**
- 写一个 `defense/gen_certs.sh`，用 OpenSSL 生成 CA + broker 证书 + 每个设备一张客户端证书
- 改 `broker/mosquitto.conf` 开启 TLS（端口 8883，关闭 1883），加 `require_certificate true` 和 `use_identity_as_username true`
- 改 `devices/simulator.py` 和 `web/app.py` 的 MQTT 连接代码，加载 cert/key
- 更新 `docker-compose.yml`，把证书目录挂进容器
- **验证 demo**：再跑一次 `mqtt_attack.sh unlock`，应该因为没有客户端证书被 broker 拒绝；Wireshark 抓包看到全是 TLS Application Data 而非明文 JSON

**D3 — Web 加固**
- 把 `web/app.py` 里的 SQL 查询改成参数化 `cursor.execute("SELECT * FROM users WHERE username=? AND password=?", (u, p))`
- 对密码做 bcrypt 哈希，不再明文存储
- 加 `Flask-WTF` 的 CSRF Token 保护
- 用 `pyotp` 实现 TOTP，第一次登录引导扫二维码，之后登录要输 6 位验证码
- **验证 demo**：再跑一次 sqlmap → 提示 not injectable；用 `admin'--` 绕过 → 失败；登录要输 TOTP

**D2 — IDS**（这是最复杂也最出彩的模块）
- 写一个 `ids/ids.py`，作为新的 Docker 容器订阅 `home/#`
- 第一阶段：**收集正常基线流量**（让系统跑 10 分钟，记录每个 topic 的发布频率、消息大小、时间分布、client_id 分布）
- 第二阶段：用 `scikit-learn` 的 Isolation Forest 训练一个简单模型，或者更简单的就用统计阈值（比如"正常 10 秒 3 条消息，超过 10 条就告警"）
- 第三阶段：加告警逻辑——检测到异常就往 `home/alert` 发告警，Web 面板订阅后红色横幅显示
- **验证 demo**：正常运行无告警；跑 `mqtt_attack.sh flood` 或 `unlock` → 30 秒内 Web 面板出现告警

**D4 + 架构整合**
- 写一个 `defense/sign_update.py`：用 `cryptography` 库生成 Ed25519 密钥对，给一个"配置更新包"（比如一个 JSON 文件）做签名
- 写一个 `defense/verify_update.py`：加载公钥，验证签名，签名错就拒绝
- **验证 demo**：正常签名的配置文件 → 加载成功；手动篡改文件内容 → 验证失败
- 同时 D 负责 **CW2 整体架构图**（防御分层图）和**报告整合**


把四个防御合起来跑一次完整 demo，确认不打架：
1. TLS broker 启动 → 设备用证书连接成功
2. 加固后的 Web 面板要 MFA 登录 → sqlmap 打不动
3. IDS 启动订阅，Dashboard 上有状态条
4. 跑一次完整攻击链 → 所有攻击被拒绝或被告警

每个防御**拿 1-2 张"前 vs 后"对比截图**：
- D1：Wireshark 对比（明文 JSON vs TLS Application Data）
- D3：sqlmap 尝试失败截图 + TOTP 登录界面截图
- D2：Web 面板上红色告警横幅截图
- D4：签名验证成功和失败的终端对比

CW2 报告也是 5 页，按模板四个小节分：
1. **Security system** (~2 页) — 整体防御架构图 + 四个防御的设计和实现
2. **Threats inferences and insights** (~1 页) — 对照 CW1 的威胁表，说明每个威胁如何被缓解
3. **Regulation and Ethical** (~1 页) — GDPR、UK PSTI Act、Cyber Resilience Act
4. **Scalability, Innovation & Enterprise** (~1 页) — 如何扩展到企业、成本、未来

5 分钟视频把 CW1 + CW2 的 demo 串起来：
- 0:00-0:30 场景介绍
- 0:30-2:00 CW1 攻击演示
- 2:00-4:30 CW2 防御演示（重点）
- 4:30-5:00 总结

- 报告 PDF 合并（CW1 + CW2）
- 视频上传 YouTube（设为 unlisted）
- GitHub repo 整理，写清楚 README
- 每人写自己的 200 字贡献说明 + 200 字 AI 使用声明
- **4/24 16:00 前提交 Moodle**

---
